# Clase 36 - Notebook 3 - algoritmo de optimización multiobjetivo por función de deseabilidad

Este notebook implementa un algoritmo de optimización alternativo al algoritmo evolutivo NSGA-II. Recordamos que el problema de optimización es el siguiente:

Minimizar simultáneamente el **contenido energético, $E$** de la corriente de aire caliente y la **humedad residual, $X_0$** del polvo. Las variables de decisión son: flujo másico del **aire a la entrada del secador, $G$** y el **radio inicial de las gotas, $r_d$** que ingresan al secador.

$$
\begin{aligned}
& \text{Minimize} \quad ContenidoEnergetico,\ X_0 \\[4pt]
& \text{subject to} \\[2pt]
\end{aligned}
$$
$$
462 \le G \le 858, \\[2pt]
3.5 \cdot 10^{-5} \le r_d \le 6.5 \cdot 10^{-5}
$$

donde:
* $G$ es el caudal de aire seco (kg/s).
* $r_d$ es el radio incial de la gota (m).


## Deseabilidad
La función de deseabilidad es uno de los métodos más usados en optimización multiobjetivo. Para cada respuesta $Y_{i}(x)$ se calcula una función de deseabilidad $d_{i}(Y_{i})$ que asigna valores entre 0 y 1 a las diferentes respuestas, donde:

- $d_{i}(Y_{i}) = 0$ indica una completa no deseabilidad de la respuesta $Y_{i}(x)$
- $d_{i}(Y_{i}) = 1$ indica una completa deseabilidad de la respuesta $Y_{i}(x)$

Las deseabilidades individuales (de cada respuesta) son combinadas usando media geométrica, obteniendo la deseabilidad general $(D)$

$D = \left(d_{1}(Y_{1})\cdot d_{2}(Y_{2}) ... d_{k}(Y_{k})\right)^{1/m}$

Donde $m$ es el número de respuestas.

Dependiendo de si una respuesta en particular debe maximizarse, minimizarse o asignarse a un valor objetivo, se utilizan diferentes funciones de deseabilidad.

**Maximización**

$$
d = \begin{cases}
0 & \text{if } y_i < L_i \\
\left(\frac{y_i - L_i}{U_i - L_i}\right)^s & \text{if } L_i \leq y_i \leq U_i \\
1 & \text{if } y_i > U_i
\end{cases}
$$

**Minimización**

$$
d = \begin{cases}
1 & \text{if } y_i < L_i \\
\left(\frac{U_i - y_i}{U_i - L_i}\right)^t & \text{if } L_i \leq y_i \leq U_i \\
0 & \text{if } y_i > U_i
\end{cases}
$$

**Target value**
$$
d = \begin{cases}
0 & \text{if } y_i < L_i \\
\left(\frac{y_i - L_i}{\bar{T}_i - L_i}\right)^s & \text{if } L_i \leq y_i \leq U_i \\
1 & \text{if } y_i = \bar{T}_i \\
\left(\frac{y_i - U_i}{\bar{T}_i - U_i}\right)^t & \text{if } \bar{T}_i \leq y_i \leq U_i \\
0 & \text{if } y_i > U_i
\end{cases}
$$

donde:

- $L_i$ es el valor mínimo aceptable para la respuesta $i$
- $U_i$ es el valor máximo aceptable para la respuesta $i$
- $T_i$ es el valor target de la respuesta $i$
- $s$ y $t$ son "pesos" que permiten controlar la tasa de variación de $d$

Las funciones que implementan este algoritmo se encuentran en la librería `spraylib`:

`build_obj2, Bounds, estimate_LU, dfa_front`

In [ ]:
# --- Preámbulo Universal para Google Colab y Entornos Locales ---
import os, sys

if 'google.colab' in sys.modules:
    REPO_DIR = '/content/DAII-SprayDrying'
    if not os.path.exists(REPO_DIR):
        !git clone https://github.com/felipehuerta17/DAII-SprayDrying.git {REPO_DIR}
    os.chdir(REPO_DIR)
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)
    %pip install -q numpy scipy pandas matplotlib pymoo casadi


Backend: idas
